# National Wetlands Inventory over the Ogallala aquifer on the 30 m Landsat/OpenET/CDL tiles: one CF file per tile

The U.S. Fish and Wildlife Service's National Wetlands Inventory (NWI) maps wetlands, deepwater habitats and
(in parts of the West) riparian areas as polygons, each with its Cowardin code. NWI is not a time series: it is
one current snapshot, put together from mapping projects flown between the 1970s and today. This notebook puts
all of it on the 30 m UTM tiles of this project's Landsat index, OpenET and CDL files and keeps, for every pixel,
the year of the imagery its mapping was made from (`image_year`), so the whole time span NWI covers stays visible.

**Output contract** (the `paper1` package reads exactly this; do not change it without changing `paper1/config.py`)

| | |
|---|---|
| File | `MyDrive/MSUGWB/NWI_wetlands_30m_tiles/<tile>/NWI_wetlands_30m_tiles_<tile>_Ogallala.nc` (rehearsal: `.../NWI_wetlands_30m_tiles/_test/<tile>/...`) |
| Grid | dims `(y, x)` on the tile's frame from the Landsat `frames.json`; `x`, `y` = pixel centres (x = x0 + (i + 0.5)·30, y = y0 − (j + 0.5)·30); grid mapping `crs` (UTM zone of the tile) |
| `roi_mask` | i1, 1 where the pixel centre lies inside the `hp_bound2010` aquifer polygon |
| `wetland_system` | u1: 0 none, 1 Marine, 2 Estuarine, 3 Riverine, 4 Lacustrine, 5 Palustrine (first letter of the NWI code); 255 outside the aquifer |
| `wetland_code` | u2 index into `attribute_table`; 0 none, 65535 outside the aquifer |
| `attribute_table` | string variable (dim `code`): the full Cowardin ATTRIBUTE code of each index, e.g. `PEM1C`, `R4SBC`; entry 0 is `""` (the table differs from tile to tile). `wetland_type_table` holds FWS's WETLAND_TYPE label of each code |
| `riparian` | u1: 1 where an NWI riparian polygon holds the pixel centre, 0 not, 255 outside |
| `nwi_mapped` | u1: 1 where a wetlands mapping project (Project_Metadata polygon, status Digital) covers the pixel, 0 not, 255 outside |
| `riparian_mapped` | u1: the same for the riparian mapping projects (riparian = 0 means "not riparian" only where this is 1) |
| `image_year` | i2: IMAGE_YR of the mapping project covering the pixel (the most recent where projects overlap); −32768 unknown, unmapped or outside |
| Global attributes | `frame` (JSON), `sources` (JSON: per FWS zip its URL, size, Last-Modified, MD5, layer names), `settings_signature`, `processing_status` (`complete`), `history`, `tile_facts` (JSON: overlap counts, repaired polygons, projects, ...), CF-1.9 |

**Source.** FWS's state geodatabases of the eight aquifer states (NE, KS, CO, TX, OK, NM, SD, WY;
`https://documentst.ecosphere.fws.gov/wetlands/data/State-Downloads/<ST>_geodatabase_wetlands.zip`, 0.6–1.8 GB
each, 7.5 GB in all on 2026-11-04). The rehearsal (and `SOURCE = "watershed"`, a fallback) reads FWS's HU8 watershed
files (`.../wetlands/downloads/watershed/HU8_<code>_Watershed.zip`, 10–140 MB); which HU8s meet a tile is asked from
USGS's Watershed Boundary Dataset. Layers read (found by the ends of their names, checked 2026-11-04):

| Role | State file (e.g. NE) | Watershed file | Fields used |
|---|---|---|---|
| wetlands and deepwater | `NE_Wetlands` | `HU8_<code>_Wetlands` | `ATTRIBUTE`, `WETLAND_TYPE` |
| riparian | `NE_Riparian` | `HU8_<code>_Riparian` (only where mapped) | – (presence) |
| wetland mapping projects | `NE_Wetlands_Project_Metadata` | `HU8_<code>_Metadata` | `STATUS`, `IMAGE_YR`, `PROJECT_NAME` |
| riparian mapping projects | `NE_Riparian_Project_Metadata` | `HU8_<code>_Riparian_Metadata` | the same |
| the file's own outline | `Nebraska` | `HU8_<code>_Watershed` | – |
| not used | `NE_Wetlands_Historic_Map_Info` (index of scanned historic map PDFs) | `HU8_<code>_Historic_Map_Info` | |

All layers are in FWS's NAD83 Albers grid (lat_0 23, lon_0 −96, standard parallels 29.5 / 45.5).

**From polygons to pixels.** A pixel takes the polygon that holds its centre (the rule of the aquifer mask, so
nothing is averaged or mixed). Polygon edges are cut into pieces of at most 500 m in FWS's grid and every corner is
projected exactly to the tile's UTM grid (pyproj); the bend this leaves out is under a millimetre. GDAL then burns
the polygons by pixel centre. No approximate warp is involved. Rules where things meet:

* every layer of a file is cut to that file's own outline first (the state, or the watershed), so two neighbouring
  files never both describe one place (state project polygons reach past the state line);
* overlapping wetland polygons (rare: 121 m² among 298,045 polygons in the rehearsal's 16 watersheds): the smaller one wins;
* overlapping mapping projects: the most recent imagery year wins; projects whose year is not stated count as mapped
  with `image_year` unknown;
* invalid polygons are repaired with shapely's `make_valid` (50 of 298,045 in the rehearsal).

**How to run**

1. *Run all* with `TEST_MODE = True`: tile `z14e02n29` (south-west Nebraska) from the 16 HU8 files that meet it
   (670 MB), into `.../_test/`. Zips already in `SOURCE_CACHE` (`MyDrive/MSUGWB/_raw_samples/NWI/`) with FWS's size
   are used instead of downloading them.
2. Set `TEST_MODE = False` and *Run all*: the eight state files are downloaded once (7.5 GB), unpacked on scratch,
   and every tile is made. Whenever the runtime dies or you stop it, run all again: finished tiles are skipped.
   The first run records the FWS files in `sources.json`; if FWS republishes one later, the run stops
   (`NEW_RELEASE = "stop"`) so that all tiles stay from one release, or remakes the affected tiles (`"rebuild"`).

**What the notebook checks**

* A download must have the length FWS states, every member taken out of a zip must match the zip's CRC, and the
  MD5 of each zip must equal the one recorded for that release.
* For every tile, 2000 random aquifer pixels are compared with a point-in-polygon test (shapely) against FWS's
  polygons as read, with the pixel centres projected by GDAL instead of pyproj. Every file is read back and
  compared before it is copied to Drive, with a checksum comparison.

**Notes**

* `wetland_system = 0` means "no NWI polygon here". Where `nwi_mapped = 0` NWI has not mapped the place, so
  absence of a wetland there is not evidence. NWI maps from aerial imagery and misses small, drier, or forested
  wetlands; it is not a jurisdictional delineation.
* The inventory keeps changing: FWS updates states as new projects arrive. `sources` in every file names the
  exact zips (with date and MD5) the tile was made from.

In [ ]:
# ═══════════════ CELL 1: install ═══════════════
# numpy, geopandas, shapely, pyproj and requests are part of Colab; pyogrio reads FWS's geodatabases.
!pip install -q rasterio netCDF4 pyogrio 2>&1 | tail -2

In [ ]:
# ═══════════════ CELL 2: mount Drive, stage the shapefile, SETTINGS ═══════════════
import os, zipfile
if not os.path.ismount('/content/drive'):
    from google.colab import drive
    drive.mount('/content/drive')

SHP_ZIP  = '/content/drive/MyDrive/high_plains_quifer.zip'
SHP_DIR  = '/content/ogallala_shp'
SHP_PATH = f'{SHP_DIR}/high_plains_quifer/hp_bound2010.shp'
if not os.path.exists(SHP_PATH):
    with zipfile.ZipFile(SHP_ZIP) as z:
        z.extractall(SHP_DIR)
print("Shapefile ready:", os.path.exists(SHP_PATH))

# ================= SETTINGS (edit these, then Run all) =================
TEST_MODE     = True       # True -> rehearsal on TEST_TILES into .../_test/ ; False -> every tile of the aquifer
TEST_TILES    = ["z14e02n29"]   # tiles of the rehearsal (z14e02n29: south-west Nebraska, Platte and Republican basins)
TEST_SOURCE   = "watershed"     # where the rehearsal reads NWI: "watershed" (FWS's HU8 files, 10-140 MB each;
                                # z14e02n29 needs 16 of them, 670 MB) or "state" (the full state files)
TEST_FRAME_PX = None       # None -> whole tile frames; or a square of this many pixels in the middle of each frame

SOURCE     = "state"       # where the full run reads NWI: "state" (one geodatabase per state, 0.6-1.8 GB each,
                           # 7.5 GB in all) or "watershed" (the HU8 files that meet each tile; for when a state
                           # file cannot be had)
STATES     = ["NE", "KS", "CO", "TX", "OK", "NM", "SD", "WY"]    # every state the aquifer reaches
ONLY_TILES = None          # None -> every tile; or e.g. ["z14e02n29"]

STATE_URL = "https://documentst.ecosphere.fws.gov/wetlands/data/State-Downloads/{st}_geodatabase_wetlands.zip"
HU8_URL   = "https://documentst.ecosphere.fws.gov/wetlands/downloads/watershed/HU8_{hu8}_Watershed.zip"
WBD_URL   = "https://hydro.nationalmap.gov/arcgis/rest/services/wbd/MapServer/4/query"
                           # USGS Watershed Boundary Dataset, HU8 layer: asked which watersheds meet a tile
                           # (the "watershed" source only; the answer is kept in hu8_by_tile.json)
HU8_LIST  = None           # None -> asked from WBD_URL; or {tile: [8-digit codes]} to name them yourself
SOURCE_CACHE    = "/content/drive/MyDrive/MSUGWB/_raw_samples/NWI"
                           # a zip found in this Drive folder with the size FWS states is used instead of downloading it
CACHE_DOWNLOADS = False    # True -> every zip downloaded is also kept in SOURCE_CACHE (the state files: 7.5 GB)
NEW_RELEASE     = "stop"   # FWS has republished a file since the first run into this folder:
                           # "stop" -> the run stops, so that all tiles come from one release;
                           # "rebuild" -> the new file is used and every tile made from the old one is made again

# ---- the output grid: the 30 m tiles of the Landsat, OpenET and CDL files ----
TILE_PX     = 5120
FRAMES_FROM = ["/content/drive/MyDrive/MSUGWB/LANDSAT_C2L2_indices_30m_monthly_tiles_v2/frames.json",
               "/content/drive/MyDrive/MSUGWB/OPENET_v2_1_monthly_30m_tiles/frames.json",
               "/content/drive/MyDrive/MSUGWB/OPENET_v2_0_monthly_30m_tiles/frames.json"]
                           # the tile frames are copied from the first of these that exists, so every tile has
                           # the same name, x and y as in those products

DENSIFY_M      = 500.0     # polygon edges are cut into pieces of at most this length (in FWS's Albers grid) before
                           # their corners are projected to the tile's UTM grid; the bend this ignores is under 1 mm
SPOT_CHECK_PX  = 2000      # random aquifer pixels per tile checked by point-in-polygon against FWS's own polygons
EDGE_TOL_M     = 0.01      # spot check: a pixel centre this close to a polygon edge is not judged (either answer is right)
COMPRESS_LEVEL = 2
CHUNK_YX       = 512
MIN_FREE_GB    = 20.0      # scratch space that must stay free (the state geodatabases unpack to about 10 GB)
DISK_MAX_WAIT_S = 2 * 3600
DOWNLOAD_TRIES = 5
KEEP_DOWNLOADS = False     # True -> keep the zip files on scratch after unpacking them
# =======================================================================

In [ ]:
# ═══════════════ CELL 3: imports, logging, paths, the variables ═══════════════
import re, gc, sys, json, glob, time, math, shutil, hashlib, logging, threading, warnings
from datetime import datetime, timezone
from email.utils import parsedate_to_datetime
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import netCDF4 as nc4
import geopandas as gpd
import pyogrio
import pyproj
import shapely
import rasterio
import rasterio.features
import rasterio.warp
from rasterio.enums import MergeAlg
from affine import Affine
import requests

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)-7s | %(message)s",
                    datefmt="%H:%M:%S", force=True)
log = logging.getLogger("nwi")
for _n in ("urllib3.connectionpool", "rasterio", "pyogrio", "fiona"):
    logging.getLogger(_n).setLevel(logging.ERROR)
warnings.filterwarnings("ignore", message=".*Measured \\(M\\) geometry types.*")

ROI_LABEL   = "Ogallala"
PRODUCT_TAG = "NWI_wetlands_30m_tiles"
DRIVE_ROOT  = "/content/drive/MyDrive/MSUGWB"
DRIVE_OUT   = f"{DRIVE_ROOT}/{PRODUCT_TAG}" + ("/_test" if TEST_MODE else "")
SCRATCH     = "/content/nwi_scratch" + ("_test" if TEST_MODE else "")
RAW_DIR     = f"{SCRATCH}/raw"                      # FWS's zip files and the geodatabases unpacked from them
NC_DIR      = f"{SCRATCH}/nc"
MASK_DIR    = f"{SCRATCH}/masks"
DISK_PATH   = "/content"
FRAMES_NAME   = "frames.json"
LEDGER_NAME   = "files_done.jsonl"
MANIFEST_NAME = "sources.json"                      # every FWS file this folder was made from: size, date, MD5
HU8_NAME      = "hu8_by_tile.json"
for _d in (DRIVE_OUT, RAW_DIR, NC_DIR, MASK_DIR):
    os.makedirs(_d, exist_ok=True)

SOURCE_KIND = TEST_SOURCE if TEST_MODE else SOURCE
if SOURCE_KIND not in ("state", "watershed"):
    raise ValueError(f"the source must be 'state' or 'watershed', not {SOURCE_KIND!r}")
if NEW_RELEASE not in ("stop", "rebuild"):
    raise ValueError(f"NEW_RELEASE must be 'stop' or 'rebuild', not {NEW_RELEASE!r}")

PIXEL = 30.0
TILE_M = TILE_PX * PIXEL
# Cowardin system = first letter of NWI's ATTRIBUTE code
SYSTEMS = {"M": 1, "E": 2, "R": 3, "L": 4, "P": 5}
SYSTEM_NAMES = {0: "none", 1: "marine", 2: "estuarine", 3: "riverine", 4: "lacustrine", 5: "palustrine"}
FILL_U1, FILL_U2, FILL_I2 = 255, 65535, -32768      # outside the aquifer polygon (image_year: also "year unknown")
# the layers read from every FWS file, found by the end of their names (state files: NE_Wetlands,
# NE_Wetlands_Project_Metadata, ...; watershed files: HU8_10250008_Wetlands, HU8_10250008_Metadata, ...)
LAYER_ENDINGS = {
    "state":     dict(wetlands="_Wetlands", riparian="_Riparian", projects="_Wetlands_Project_Metadata",
                      riparian_projects="_Riparian_Project_Metadata", historic="_Wetlands_Historic_Map_Info"),
    "watershed": dict(wetlands="_Wetlands", riparian="_Riparian", projects="_Metadata",
                      riparian_projects="_Riparian_Metadata", historic="_Historic_Map_Info", boundary="_Watershed"),
}
PROJECT_FIELDS = ["PROJECT_NAME", "STATUS", "IMAGE_YR", "IMAGE_DATE", "SOURCE_TYPE"]


class PipelineStop(RuntimeError):
    """Clean stop: everything on Drive is safe, re-running resumes."""


class DataIntegrityError(RuntimeError):
    """Something that must never reach the output was detected."""


def utcnow():
    return datetime.now(timezone.utc).replace(tzinfo=None)


def iso(dt):
    return dt.strftime("%Y-%m-%dT%H:%M:%S.%f")[:-3] + "Z"

In [ ]:
# ═══════════════ CELL 4: disk floor, verified Drive copy, logs ═══════════════
_LOG_LOCK   = threading.Lock()
_DRIVE_LOCK = threading.Lock()
DRIVE_SETTLE_S = 10                      # let the Drive FUSE mount settle before re-reading
TIMES = {}                               # seconds spent per stage (reported at the end of the run)
TOTALS = {}                              # counters (bytes downloaded, files, records) for the summary


class timed:
    """with timed("download"): ...  adds the elapsed seconds to TIMES["download"]."""

    def __init__(self, name):
        self.name = name

    def __enter__(self):
        self.t0 = time.time()
        return self

    def __exit__(self, *exc):
        TIMES[self.name] = TIMES.get(self.name, 0.0) + time.time() - self.t0
        return False


def log_events(name, entries):
    """Append JSON lines to a log on Drive in one write + fsync (survives a dead runtime)."""
    if not entries:
        return
    stamp = iso(utcnow())
    text = "".join(json.dumps(dict(e, logged=stamp), default=str) + "\n" for e in entries)
    with _LOG_LOCK:
        with open(os.path.join(DRIVE_OUT, name), "a") as f:
            f.write(text)
            f.flush()
            os.fsync(f.fileno())


def human(n):
    n = float(n)
    for u in ("B", "KB", "MB", "GB", "TB"):
        if abs(n) < 1024:
            return f"{n:.1f} {u}"
        n /= 1024
    return f"{n:.1f} PB"


# ---------- 20 GB scratch floor ----------
def free_gb():
    return shutil.disk_usage(DISK_PATH).free / 1024 ** 3


def ensure_disk_free(context="", extra_gb=0.0):
    """Block until MIN_FREE_GB + extra_gb is free. Scratch only ever holds the file being built,
    so a shortfall normally means Drive's local upload cache is still draining: wait for it
    rather than fail. Raises PipelineStop if it never clears."""
    need = MIN_FREE_GB + max(0.0, extra_gb)
    t0, warned = time.time(), False
    while True:
        free = free_gb()
        if free >= need:
            if warned:
                log.info("Disk OK again: %.1f GB free (need %.1f)", free, need)
            return free
        if not warned:
            warned = True
            log.warning("LOW DISK %.1f < %.1f GB (%s) - waiting for Drive to upload", free, need, context)
        if time.time() - t0 > DISK_MAX_WAIT_S:
            raise PipelineStop(f"Free space stayed at {free:.1f} GB (< {need:.1f} GB) for "
                               f"{DISK_MAX_WAIT_S // 60} min ({context}). Nothing is lost - re-run to resume.")
        time.sleep(60)


# ---------- verified copies to / from Drive ----------
def md5sum(path, block=10 * 1024 * 1024):
    h = hashlib.md5()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(block)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()


def robust_drive_copy(src, dst, max_retries=3):
    """Copy src to Drive under a temporary name, re-read it, compare size + MD5, then
    rename over dst. A runtime that dies mid-copy leaves the previous good dst intact."""
    if os.path.abspath(src) == os.path.abspath(dst):
        raise ValueError(f"robust_drive_copy: source and destination are the same file ({src})")
    with _DRIVE_LOCK, timed("Drive copy"):
        size = os.path.getsize(src)
        ensure_disk_free("Drive copy", extra_gb=size / 1024 ** 3)   # FUSE caches the upload locally
        want = md5sum(src)
        tmp = dst + ".tmp"
        os.makedirs(os.path.dirname(dst) or ".", exist_ok=True)
        last = None
        for attempt in range(1, max_retries + 1):
            try:
                with open(src, "rb") as fs, open(tmp, "wb") as fd:
                    shutil.copyfileobj(fs, fd, length=10 * 1024 * 1024)
                    fd.flush()
                    os.fsync(fd.fileno())
                ok = False
                for _ in range(4):           # a just-closed FUSE file can return stale bytes
                    time.sleep(DRIVE_SETTLE_S)
                    try:
                        if os.path.getsize(tmp) == size and md5sum(tmp) == want:
                            ok = True
                            break
                    except OSError as e:
                        last = e
                if not ok:
                    raise IOError("size or MD5 mismatch after settle")
                os.replace(tmp, dst)
                log.info("Drive copy verified (%s): %s", human(size), os.path.basename(dst))
                return True
            except Exception as e:
                last = e
                log.warning("Drive copy attempt %d/%d failed: %s", attempt, max_retries, e)
                try:
                    if os.path.exists(tmp):
                        os.remove(tmp)
                except OSError:
                    pass
                if attempt < max_retries:
                    time.sleep(10 * attempt)
        raise RuntimeError(f"Drive copy of {os.path.basename(src)} failed after {max_retries} attempts: {last}")


def restore_from_drive(src, dst):
    """Bring a Drive file back to scratch (size-checked, atomic)."""
    size = os.path.getsize(src)
    ensure_disk_free("restore from Drive", extra_gb=size / 1024 ** 3)
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    tmp = dst + ".part"
    shutil.copyfile(src, tmp)
    if os.path.getsize(tmp) != size:
        os.remove(tmp)
        raise IOError(f"restore of {os.path.basename(src)} is incomplete")
    os.replace(tmp, dst)
    log.info("Restored %s from Drive (%s)", os.path.basename(src), human(size))

In [ ]:
# ═══════════════ CELL 5: aquifer polygon and the tiles (the frames of the Landsat / OpenET / CDL files) ═══════════════
_ROI_GDF = gpd.read_file(SHP_PATH)
if _ROI_GDF.crs is None:
    raise ValueError(f"{SHP_PATH} has no CRS")
_POLY = {}


def roi_polygon(crs):
    """The aquifer polygon in a given CRS (one copy per CRS). The shapefile's parts are merged once,
    in its own CRS; the merged outline is then reprojected, which is quick."""
    if None not in _POLY:
        g = _ROI_GDF.geometry
        u = g.union_all() if hasattr(g, "union_all") else g.unary_union
        _POLY[None] = u if u.is_valid else u.buffer(0)
    if crs not in _POLY:
        u = gpd.GeoSeries([_POLY[None]], crs=_ROI_GDF.crs).to_crs(crs).iloc[0]
        _POLY[crs] = u if u.is_valid else u.buffer(0)
    return _POLY[crs]

def tile_key(crs, i, j):
    return f"z{int(crs.split(':')[1]) % 100:02d}e{i:02d}n{j:02d}"

def _mask_path(key):
    return os.path.join(MASK_DIR, f"{key}.npy")

def _rasterize(F):
    return rasterio.features.geometry_mask([roi_polygon(F["crs"])], out_shape=(F["h"], F["w"]), invert=True,
                                           all_touched=False,          # pixel centre inside the polygon
                                           transform=Affine(PIXEL, 0.0, F["x0"], 0.0, -PIXEL, F["y0"]))

def _save_mask(key, mask):
    tmp = _mask_path(key) + ".part"
    with open(tmp, "wb") as f:
        np.save(f, np.packbits(mask))
    os.replace(tmp, _mask_path(key))

def frame_mask(key, F):
    """True where the pixel centre lies inside the aquifer polygon (kept on scratch, 1 bit per pixel)."""
    if os.path.exists(_mask_path(key)):
        bits = np.load(_mask_path(key))
        if bits.size * 8 >= F["h"] * F["w"]:
            mask = np.unpackbits(bits)[:F["h"] * F["w"]].reshape(F["h"], F["w"]).astype(bool)
            if int(mask.sum()) == F["n_roi"]:
                return mask
    mask = _rasterize(F)
    if int(mask.sum()) != F["n_roi"]:
        raise DataIntegrityError(f"{key}: the aquifer mask has {int(mask.sum())} pixels, its frame says {F['n_roi']}. "
                                 "The shapefile differs from the one the frames were made with.")
    _save_mask(key, mask)
    return mask

def make_frame(key, crs, ox, oy, i, j):
    """Frame of tile (i, j) of a zone whose lattice has its pixel edges at ox, oy (mod 30 m): the
    tile cut to the rectangle that bounds its aquifer pixels. None when it holds no aquifer pixel."""
    left, top = ox + i * TILE_M, oy + (j + 1) * TILE_M
    box = shapely.geometry.box(left, top - TILE_M, left + TILE_M, top)
    if not roi_polygon(crs).intersects(box):
        return None
    F = dict(crs=crs, x0=left, y0=top, w=TILE_PX, h=TILE_PX)
    mask = _rasterize(F)
    if not mask.any():
        return None
    rows, cols = np.flatnonzero(mask.any(axis=1)), np.flatnonzero(mask.any(axis=0))
    r0, r1, c0, c1 = int(rows[0]), int(rows[-1]) + 1, int(cols[0]), int(cols[-1]) + 1
    sub = np.ascontiguousarray(mask[r0:r1, c0:c1])
    F = dict(crs=crs, x0=left + c0 * PIXEL, y0=top - r0 * PIXEL, w=c1 - c0, h=r1 - r0, n_roi=int(sub.sum()))
    _save_mask(key, sub)
    return F


def zone_tiles(only=None):
    """Tiles of every UTM zone the aquifer reaches, used only when no frames.json of the Landsat or
    OpenET files is found: the tiles of zone z that meet the part of the aquifer within 1.5 degrees of
    the zone's own longitudes. The Landsat files' tiles depend on where Landsat scenes lie and can
    differ a little at zone edges; their frames.json is therefore preferred. only: a set of tile names
    to keep (the rehearsal needs only its own tiles)."""
    poly = roi_polygon("EPSG:4326")
    cand = {}
    for zone in range(1, 61):
        lon0 = -180 + 6 * (zone - 1)
        band = shapely.geometry.box(lon0 - 1.5, -80, lon0 + 6 + 1.5, 84)
        part = poly.intersection(band)
        if part.is_empty:
            continue
        crs = f"EPSG:{32600 + zone}"
        part_utm = gpd.GeoSeries([part], crs="EPSG:4326").to_crs(crs).iloc[0]
        x0, y0, x1, y1 = part_utm.bounds
        ox = oy = 15.0                                   # Landsat's lattice: pixel edges at 15 m mod 30
        for i in range(math.floor((x0 - ox) / TILE_M), math.floor((x1 - ox) / TILE_M) + 1):
            for j in range(math.floor((y0 - oy) / TILE_M), math.floor((y1 - oy) / TILE_M) + 1):
                key = tile_key(crs, i, j)
                if only is None or key in only:
                    cand[key] = (crs, ox, oy, i, j)
    return cand


def frames_tag():
    return dict(tile_px=TILE_PX, pixel=PIXEL, test_frame_px=TEST_FRAME_PX if TEST_MODE else None)


def get_frames():
    """{tile: frame}. Taken from the first FRAMES_FROM file that exists (so the tiles are those of the
    Landsat / OpenET / CDL files), otherwise made here; stored in this product's folder either way."""
    path = os.path.join(DRIVE_OUT, FRAMES_NAME)
    if os.path.exists(path):
        with open(path) as f:
            doc = json.load(f)
        if doc.get("made_with") == frames_tag() and (not TEST_MODE or set(TEST_TILES) <= set(doc["frames"])):
            log.info("Tile frames from %s (%s)", path, doc.get("origin", "?"))
            return {k: v for k, v in doc["frames"].items() if v}
        if not TEST_MODE:
            raise PipelineStop(f"{FRAMES_NAME} in {DRIVE_OUT} was made with other settings ({doc.get('made_with')}); "
                               "the files there would not fit those of this run. Use an empty output folder.")
    frames, origin = None, None
    for src in FRAMES_FROM:
        if os.path.exists(src):
            with open(src) as f:
                doc = json.load(f)
            tag = doc.get("made_with", {})
            if tag.get("tile_px") == TILE_PX and tag.get("pixel") in (None, PIXEL) and not tag.get("test_frame_px"):
                frames, origin = {k: v for k, v in doc["frames"].items() if v}, src
                break
    if frames is None:
        origin = "made here from the UTM zones (no frames.json of the Landsat or OpenET files was found)"
        frames = {}
        for key, args in sorted(zone_tiles(set(TEST_TILES) if TEST_MODE else None).items()):
            F = make_frame(key, *args)
            if F:
                frames[key] = F
    if TEST_MODE:
        frames = {k: F for k, F in frames.items() if k in set(TEST_TILES)}
    if TEST_MODE and TEST_FRAME_PX:            # the rehearsal: a square in the middle of each tile's frame
        small = {}
        for key, F in frames.items():
            n = min(TEST_FRAME_PX, F["w"], F["h"])
            c0, r0 = (F["w"] - n) // 2, (F["h"] - n) // 2
            G = dict(crs=F["crs"], x0=F["x0"] + c0 * PIXEL, y0=F["y0"] - r0 * PIXEL, w=n, h=n)
            m = _rasterize(G)
            if m.any():
                small[key] = dict(G, n_roi=int(m.sum()))
        frames = small
    local = os.path.join(NC_DIR, FRAMES_NAME)
    with open(local, "w") as f:
        json.dump(dict(origin=origin, made_with=frames_tag(), created=iso(utcnow()), frames=frames), f, indent=1)
    robust_drive_copy(local, path)
    os.remove(local)
    log.info("Tile frames: %d tiles, %s", len(frames), origin)
    return frames

In [ ]:
# ═══════════════ CELL 6: FWS's files: which ones, their release, download or cache, unpack, layers ═══════════════
HTTP = requests.Session()
HTTP.headers["User-Agent"] = "MSU groundwater project NWI notebook (python-requests)"
NET_WAITS_S = [30, 60, 120, 240, 300, 300, 300]     # pauses between attempts when a server cannot be reached


def _use_ipv4_only():
    """'Network is unreachable' can mean the runtime has no IPv6 route while a host is tried over IPv6
    first; from then on only IPv4 addresses are used."""
    import urllib3.util.connection as uc
    if getattr(uc, "HAS_IPV6", False):
        uc.HAS_IPV6 = False
        log.warning("Network unreachable - using IPv4 only from now on")


def http_head(url):
    """The server's answer about one file (status and headers). A server that cannot be reached is asked
    again after growing pauses (about 20 minutes in all) and only then is the run stopped. A missing
    file (404) is an answer, not an error."""
    last = None
    for attempt, pause in enumerate([0] + NET_WAITS_S):
        if pause:
            log.warning("%s could not be reached (%s) - trying again in %d s", url.split("/")[2], str(last)[:120], pause)
            time.sleep(pause)
        try:
            r = HTTP.head(url, timeout=60, allow_redirects=True)
            if r.status_code >= 500:
                raise IOError(f"HTTP {r.status_code}")
            return r
        except (IOError, requests.RequestException) as e:
            last = e
            if "unreachable" in str(e).lower():
                _use_ipv4_only()
    raise PipelineStop(f"{url} could not be reached ({str(last)[:200]}). Everything on Drive is safe - run again later.")


def _save_json(doc, name):
    local = os.path.join(NC_DIR, name)
    with open(local, "w") as f:
        json.dump(doc, f, indent=1, sort_keys=True)
    robust_drive_copy(local, os.path.join(DRIVE_OUT, name))
    os.remove(local)


# ---------- which files a run needs ----------
def hu8_by_tile(frames):
    """{tile: [HU8 codes]} of the watersheds that meet each tile's frame (watershed source only): HU8_LIST
    when given, else asked from USGS's Watershed Boundary Dataset, once per tile (kept in hu8_by_tile.json)."""
    path = os.path.join(DRIVE_OUT, HU8_NAME)
    known = {}
    if os.path.exists(path):
        with open(path) as f:
            known = json.load(f)
    if HU8_LIST:
        known.update({k: sorted(v) for k, v in HU8_LIST.items()})
    new = 0
    for key, F in sorted(frames.items()):
        if key in known:
            continue
        pad = 2000.0                    # metres around the frame, so that no watershed at its edge is missed
        env = ",".join(str(v) for v in (F["x0"] - pad, F["y0"] - F["h"] * PIXEL - pad,
                                         F["x0"] + F["w"] * PIXEL + pad, F["y0"] + pad))
        params = dict(geometry=env, geometryType="esriGeometryEnvelope", inSR=pyproj.CRS(F["crs"]).to_epsg(),
                      spatialRel="esriSpatialRelIntersects", outFields="huc8", returnGeometry="false", f="json")
        last = None
        for attempt, pause in enumerate([0] + NET_WAITS_S):
            if pause:
                log.warning("USGS's watershed service: %s - trying again in %d s", str(last)[:120], pause)
                time.sleep(pause)
            try:
                r = HTTP.get(WBD_URL, params=params, timeout=120)
                r.raise_for_status()
                doc = r.json()
                if "features" not in doc:
                    raise IOError(str(doc.get("error", doc))[:200])
                break
            except (IOError, ValueError, requests.RequestException) as e:
                last = e
        else:
            raise PipelineStop(f"USGS's watershed service could not say which HU8s meet {key} ({str(last)[:200]}). "
                               "Run again later, or name them in HU8_LIST.")
        codes = sorted({str(f["attributes"]["huc8"]) for f in doc["features"]})
        if not codes:
            raise PipelineStop(f"USGS's watershed service names no HU8 for tile {key}.")
        known[key] = codes
        new += 1
    if new:
        _save_json(known, HU8_NAME)
        log.info("HU8 watersheds of %d tile(s) from USGS's Watershed Boundary Dataset", new)
    return {k: known[k] for k in frames}


def wanted_files(frames):
    """({zip name: url} of every FWS file the run reads, {tile: [zip names]}). With the state source every
    tile may read every state; which states really meet a tile is decided from their outlines."""
    if SOURCE_KIND == "state":
        files = {f"{st}_geodatabase_wetlands.zip": STATE_URL.format(st=st) for st in STATES}
        return files, {k: sorted(files) for k in frames}
    per_tile = {k: [f"HU8_{h}_Watershed.zip" for h in v] for k, v in hu8_by_tile(frames).items()}
    files = {n: HU8_URL.format(hu8=n.split("_")[1]) for names in per_tile.values() for n in names}
    return files, per_tile


# ---------- the release: what FWS offers now against what this folder was made from ----------
def load_manifest():
    path = os.path.join(DRIVE_OUT, MANIFEST_NAME)
    if not os.path.exists(path):
        return {}
    with open(path) as f:
        return json.load(f)["files"]


def save_manifest(man):
    _save_json(dict(product=PRODUCT_TAG, source=SOURCE_KIND, updated=iso(utcnow()), files=man), MANIFEST_NAME)


def check_release(files):
    """Every needed file's size and date as FWS states them now, held against sources.json (the files
    this folder was made from). Returns that manifest: {zip name: dict(url, size, last_modified, md5)};
    the MD5 is filled in once the file has been read."""
    with ThreadPoolExecutor(8) as ex:
        heads = dict(zip(files, ex.map(http_head, files.values())))
    bad = {n: r.status_code for n, r in heads.items() if r.status_code != 200}
    if bad:
        raise PipelineStop("FWS does not offer " + ", ".join(f"{n} (HTTP {s})" for n, s in sorted(bad.items())) +
                           ("" if SOURCE_KIND == "watershed" else ". SOURCE = 'watershed' reads the HU8 files instead."))
    man, changed, dirty = load_manifest(), [], False
    for n, r in sorted(heads.items()):
        now = dict(url=files[n], size=int(r.headers["Content-Length"]), last_modified=r.headers.get("Last-Modified", ""))
        old = man.get(n)
        if old is None:
            man[n], dirty = dict(now, md5=None), True
        elif (old["size"], old["last_modified"]) != (now["size"], now["last_modified"]):
            changed.append(f"{n} ({old['last_modified']}, {human(old['size'])} -> {now['last_modified']}, {human(now['size'])})")
            if NEW_RELEASE == "rebuild":
                man[n] = dict(now, md5=None, previous={k: old.get(k) for k in ("size", "last_modified", "md5")})
                dirty = True
    if changed and NEW_RELEASE == "stop":
        raise PipelineStop("FWS has republished " + "; ".join(changed) + ". The tiles in this folder come from the "
                           "earlier files. NEW_RELEASE = 'rebuild' makes them again from the new ones (or use a new "
                           "output folder). Nothing was changed.")
    if changed:
        log.warning("FWS has republished %s; the tiles made from the earlier files are made again", "; ".join(changed))
    if dirty:
        save_manifest(man)
    return man


# ---------- the files themselves ----------
def download(url, name, size):
    """A file on scratch, complete: streamed in one request and continued with byte ranges after a break;
    its length must equal the server's."""
    path = os.path.join(RAW_DIR, name)
    if os.path.exists(path) and os.path.getsize(path) == size:
        return path
    ensure_disk_free(f"download {name}", extra_gb=size / 1024 ** 3)
    part, t0 = path + ".part", time.time()
    for attempt in range(1, DOWNLOAD_TRIES + 1):
        have = os.path.getsize(part) if os.path.exists(part) else 0
        try:
            hdr = {"Range": f"bytes={have}-"} if have else {}
            with HTTP.get(url, headers=hdr, stream=True, timeout=120) as r:
                if have and r.status_code != 206:
                    have = 0                     # the server ignored the range: start again
                elif r.status_code not in (200, 206):
                    raise IOError(f"HTTP {r.status_code}")
                with open(part, "ab" if have else "wb") as f:
                    for chunk in r.iter_content(8 * 1024 * 1024):
                        f.write(chunk)
            if os.path.getsize(part) == size:
                os.replace(part, path)
                TOTALS["bytes"] = TOTALS.get("bytes", 0) + size
                log.info("Downloaded %s: %s in %.0f s", name, human(size), time.time() - t0)
                return path
            raise IOError(f"{os.path.getsize(part)} of {size} bytes")
        except (IOError, requests.RequestException) as e:
            log.warning("Download of %s broke off (%s), attempt %d of %d", name, str(e)[:120], attempt, DOWNLOAD_TRIES)
            if "unreachable" in str(e).lower():
                _use_ipv4_only()
            time.sleep(NET_WAITS_S[min(attempt, len(NET_WAITS_S)) - 1])
    raise PipelineStop(f"{name} could not be downloaded completely. Everything on Drive is safe - run again.")


def fetch_zip(name, entry):
    """(path, "cache" | "download") of a zip of the release: taken from SOURCE_CACHE when it holds the
    file at FWS's size, otherwise downloaded. Its MD5 must equal the one recorded for this release; the
    first time it is recorded."""
    cached = os.path.join(SOURCE_CACHE, name) if SOURCE_CACHE else None
    if cached and os.path.exists(cached) and os.path.getsize(cached) == entry["size"]:
        path, origin = cached, "cache"
    else:
        with timed("download"):
            path, origin = download(entry["url"], name, entry["size"]), "download"
    with timed("MD5"):
        md5 = md5sum(path)
    if entry.get("md5") and entry["md5"] != md5:
        if origin == "download":
            os.remove(path)
        raise PipelineStop(f"{name} ({origin}) has MD5 {md5}, but this folder was made from a file of the same size "
                           f"and date with MD5 {entry['md5']}. Either the file changed at FWS without a new date, or "
                           "the cached copy is not the one used before. Nothing was changed.")
    entry["md5"] = md5
    if CACHE_DOWNLOADS and cached and origin == "download":
        robust_drive_copy(path, cached)
    log.info("%s: %s from the %s, MD5 %s", name, human(entry["size"]), origin, md5)
    return path, origin


def unpack(name, zip_path, md5):
    """The geodatabase inside a zip, unpacked to RAW_DIR/<name>/. zipfile checks every member against the
    CRC the zip stores, so a damaged file cannot pass. A folder unpacked earlier from the same file (same
    MD5) is used again. (The watershed zips hold shapefile copies of the same layers; only the geodatabase
    is unpacked, so both sources are read the same way.)"""
    dest = os.path.join(RAW_DIR, name[:-4])
    stamp = os.path.join(dest, ".unpacked_from_md5")
    if os.path.exists(stamp):
        with open(stamp) as f:
            if f.read().strip() == md5:
                return dest
    shutil.rmtree(dest, ignore_errors=True)
    with zipfile.ZipFile(zip_path) as z:
        members = [i for i in z.infolist() if not i.is_dir() and ".gdb/" in i.filename]
        if not members:
            raise DataIntegrityError(f"{name} holds no geodatabase")
        ensure_disk_free(f"unpack {name}", extra_gb=sum(i.file_size for i in members) / 1024 ** 3)
        root = os.path.realpath(dest)
        try:
            with timed("unpack"):
                for i in members:
                    target = os.path.realpath(os.path.join(dest, i.filename))
                    if not target.startswith(root + os.sep):
                        raise DataIntegrityError(f"{name}: member {i.filename} would land outside {dest}")
                    os.makedirs(os.path.dirname(target), exist_ok=True)
                    with z.open(i) as s, open(target, "wb") as d:
                        shutil.copyfileobj(s, d, 16 * 1024 * 1024)    # raises BadZipFile on a CRC mismatch
        except zipfile.BadZipFile as e:
            shutil.rmtree(dest, ignore_errors=True)
            raise PipelineStop(f"{name} is damaged ({e}). Delete it from the cache (if it came from there) and run again.")
    with open(stamp, "w") as f:
        f.write(md5)
    return dest


def open_source(name, folder, entry):
    """One FWS file ready to read: its geodatabase, its layers by role (found by the ends of their names),
    its CRS, and its own outline (the state, or the HU8 watershed)."""
    gdbs = sorted(glob.glob(os.path.join(folder, "**", "*.gdb"), recursive=True))
    if len(gdbs) != 1:
        raise DataIntegrityError(f"{name}: {len(gdbs)} geodatabases inside; one was expected")
    gdb = gdbs[0]
    names = [str(n) for n in pyogrio.list_layers(gdb)[:, 0]]
    left, roles = list(names), {}
    for role, end in sorted(LAYER_ENDINGS[SOURCE_KIND].items(), key=lambda kv: -len(kv[1])):   # longest ending first
        hits = [n for n in left if n.endswith(end)]
        if len(hits) > 1:
            raise DataIntegrityError(f"{name}: layers {hits} all end in {end}")
        if hits:
            roles[role] = hits[0]
            left.remove(hits[0])
    if SOURCE_KIND == "state":                  # the state's outline is the one layer left, e.g. "Nebraska"
        if len(left) != 1:
            raise DataIntegrityError(f"{name}: the state outline cannot be told among the layers {left}")
        roles["boundary"] = left[0]
    for role in ("wetlands", "projects", "boundary"):
        if role not in roles:
            raise DataIntegrityError(f"{name}: no {role} layer among {names}")
    info = {role: pyogrio.read_info(gdb, layer=layer) for role, layer in roles.items()}
    crs = pyproj.CRS.from_user_input(info["wetlands"]["crs"])
    for role, i in info.items():
        if not pyproj.CRS.from_user_input(i["crs"]).equals(crs, ignore_axis_order=True):
            raise DataIntegrityError(f"{name}: layer {roles[role]} is not in the CRS of the wetlands layer")
    need = {"wetlands": ["ATTRIBUTE", "WETLAND_TYPE"], "riparian": ["ATTRIBUTE"],
            "projects": PROJECT_FIELDS, "riparian_projects": PROJECT_FIELDS}
    for role, fields in need.items():
        if role in roles and not set(fields) <= set(info[role]["fields"]):
            raise DataIntegrityError(f"{name}: layer {roles[role]} lacks {sorted(set(fields) - set(info[role]['fields']))}")
    outline = pyogrio.read_dataframe(gdb, layer=roles["boundary"], columns=[]).geometry.values
    outline = shapely.union_all(polygonal(outline))
    shapely.prepare(outline)
    return dict(name=name, url=entry["url"], size=entry["size"], last_modified=entry["last_modified"], md5=entry["md5"],
                gdb=gdb, layers=roles, crs=crs, outline=outline,
                n_features={role: int(i["features"]) for role, i in info.items()})


def polygonal(geoms):
    """The polygonal part of each geometry: invalid ones are repaired with shapely.make_valid, and what the
    repair turns into lines or points is dropped (it holds no pixel centre)."""
    geoms = np.asarray(geoms, dtype=object)
    bad = ~shapely.is_valid(geoms)
    if bad.any():
        fixed = shapely.make_valid(geoms[bad])
        for k, g in zip(np.flatnonzero(bad), fixed):
            if g.geom_type not in ("Polygon", "MultiPolygon"):
                parts = [p for p in shapely.get_parts(g) if p.geom_type in ("Polygon", "MultiPolygon")]
                g = shapely.union_all(parts) if parts else shapely.Polygon()
            geoms[k] = g
    return geoms


def prepare_sources(names, man):
    """{zip name: source} for the files the tiles still to be made need: fetched, checked, unpacked, opened.
    The MD5 of each file is recorded in sources.json before the first tile is written."""
    out, before = {}, {n: man[n].get("md5") for n in names}
    for name in sorted(names):
        path, origin = fetch_zip(name, man[name])
        folder = unpack(name, path, man[name]["md5"])
        if origin == "download" and not KEEP_DOWNLOADS:
            os.remove(path)
        out[name] = open_source(name, folder, man[name])
    if any(man[n]["md5"] != before[n] for n in names):
        save_manifest(man)
    return out

In [ ]:
# ═══════════════ CELL 7: one tile: read FWS's polygons, burn them by pixel centre, check, write the file ═══════════════
def frame_transform(F):
    return Affine(PIXEL, 0.0, F["x0"], 0.0, -PIXEL, F["y0"])


def frame_outline(F, crs):
    """The frame's outline as a polygon in another CRS (65 points on each edge, as the projections differ)."""
    x = np.linspace(F["x0"], F["x0"] + F["w"] * PIXEL, 65)
    y = np.linspace(F["y0"], F["y0"] - F["h"] * PIXEL, 65)
    rx = np.concatenate([x, np.full(64, x[-1]), x[::-1][1:], np.full(63, x[0])])
    ry = np.concatenate([np.full(65, y[0]), y[1:], np.full(64, y[-1]), y[::-1][1:-1]])
    tx, ty = pyproj.Transformer.from_crs(F["crs"], crs, always_xy=True).transform(rx, ry)
    return shapely.Polygon(np.column_stack([tx, ty]))


def sources_of_tile(F, names, srcs):
    """The sources whose own outline (state or watershed) meets the frame."""
    return [srcs[n] for n in names if srcs[n]["outline"].intersects(frame_outline(F, srcs[n]["crs"]).buffer(100.0))]


def read_layer(src, role, F, columns, stats):
    """The polygons of one layer of one source that meet the frame, in the source's own CRS, cut to the
    source's own outline (so that two neighbouring files never both describe one place): a table with
    the columns asked for and `geometry_src`. Invalid polygons are repaired (and counted)."""
    if role not in src["layers"]:
        return None
    look = frame_outline(F, src["crs"]).buffer(100.0)
    with timed("read polygons"):
        gdf = pyogrio.read_dataframe(src["gdb"], layer=src["layers"][role], columns=columns, bbox=look.bounds)
    gdf = gdf[gdf.geometry.notna()].reset_index(drop=True)
    g = np.asarray(gdf.geometry.array, dtype=object)
    n_bad = int((~shapely.is_valid(g)).sum()) if g.size else 0
    g = polygonal(g)
    outside = ~shapely.within(g, src["outline"]) if g.size else np.zeros(0, bool)
    if outside.any():                            # cut what reaches past the file's own state or watershed
        g[outside] = polygonal(shapely.intersection(g[outside], src["outline"]))
    keep = (~shapely.is_empty(g) & shapely.intersects(g, look)) if g.size else np.zeros(0, bool)
    table = pd.DataFrame(gdf.loc[keep, columns]).reset_index(drop=True)
    table["geometry_src"] = g[keep]
    stats["repaired_polygons"] = stats.get("repaired_polygons", 0) + n_bad
    stats["cut_to_outline"] = stats.get("cut_to_outline", 0) + int(outside.sum())
    stats.setdefault("features_read", {}).setdefault(src["name"], {})[role] = int(keep.sum())
    return table


def to_tile(geoms, src_crs, F):
    """Polygons projected to the tile's grid: edges cut into pieces of at most DENSIFY_M in the source's
    CRS, then every corner projected exactly with pyproj."""
    tr = pyproj.Transformer.from_crs(src_crs, F["crs"], always_xy=True)
    g = shapely.segmentize(np.asarray(geoms, dtype=object), DENSIFY_M)
    return shapely.transform(g, lambda xy: np.column_stack(tr.transform(xy[:, 0], xy[:, 1])))


def collect(srcs, role, F, columns, stats):
    """The polygons of one role from every source of the tile in one table: the asked columns,
    `geometry_src` (source CRS), `geometry_tile` (tile CRS), `source` and `src_crs`."""
    parts = []
    for s in srcs:
        t = read_layer(s, role, F, columns, stats)
        if t is None or not len(t):
            continue
        with timed("project polygons"):
            t["geometry_tile"] = to_tile(t["geometry_src"].values, s["crs"], F)
        t["source"], t["src_crs"] = s["name"], s["crs"].to_wkt()
        parts.append(t)
    if not parts:
        return pd.DataFrame(columns=columns + ["geometry_src", "geometry_tile", "source", "src_crs"])
    return pd.concat(parts, ignore_index=True)


def burn(geoms, values, F, dtype, fill, merge=MergeAlg.replace):
    """Polygons on the frame by pixel centre (GDAL's rasterizer with all_touched off: a pixel takes a
    polygon's value when its centre lies inside it). With MergeAlg.replace the polygon burnt last wins."""
    geoms, values = list(geoms), list(values)
    if not geoms:
        return np.full((F["h"], F["w"]), fill, dtype=dtype)
    with timed("burn"):
        return rasterio.features.rasterize(zip(geoms, values), out_shape=(F["h"], F["w"]), transform=frame_transform(F),
                                           fill=fill, all_touched=False, dtype=dtype, merge_alg=merge)


def project_year(v):
    try:
        y = int(v)
    except (TypeError, ValueError):
        return None
    return y if 1900 <= y <= 2100 else None


def is_digital(status):
    return str(status).strip().lower() == "digital"


def make_tile(key, F, srcs, mask):
    """Every variable of one tile on its frame (before the outside-aquifer fill), the code tables, the
    facts the file records, and the polygon tables in burn order (for the spot check)."""
    stats, out, polys = {}, {}, {}
    # ---- wetlands and deepwater habitats: the Cowardin code under each pixel centre ----
    W = collect(srcs, "wetlands", F, ["ATTRIBUTE", "WETLAND_TYPE"], stats)
    W["ATTRIBUTE"] = W["ATTRIBUTE"].astype(str).str.strip()
    odd = sorted({a for a in W["ATTRIBUTE"] if not a or a[0] not in SYSTEMS})
    if odd:
        raise DataIntegrityError(f"{key}: NWI codes that do not start with a Cowardin system letter (M E R L P): {odd[:20]}")
    codes = [""] + sorted(set(W["ATTRIBUTE"]))
    if len(codes) >= FILL_U2:
        raise DataIntegrityError(f"{key}: {len(codes)} distinct NWI codes do not fit 16 bits")
    index = {a: k for k, a in enumerate(codes)}
    types = {a: "; ".join(sorted({str(t) for t in g.dropna()})) for a, g in W.groupby("ATTRIBUTE")["WETLAND_TYPE"]}
    W["code"] = W["ATTRIBUTE"].map(index).astype("u2")
    W["area"] = shapely.area(W["geometry_src"].values.astype(object)) if len(W) else np.zeros(0)
    # Where polygons overlap, the smaller one is burnt last and wins (the more specific feature, e.g. a
    # pond lying in a larger wetland that was not cut out of it). The order is fixed, so reruns are equal.
    W = W.sort_values(["area", "ATTRIBUTE"], ascending=[False, True], kind="mergesort").reset_index(drop=True)
    out["wetland_code"] = burn(W["geometry_tile"], W["code"], F, "u2", 0)
    count = burn(W["geometry_tile"], [1] * len(W), F, "u1", 0, MergeAlg.add)
    other = burn(W["geometry_tile"][::-1], W["code"][::-1], F, "u2", 0)          # the larger one on top instead
    stats["overlap_pixels"] = int(np.count_nonzero(mask & (count > 1)))
    stats["overlap_pixels_with_different_code"] = int(np.count_nonzero(mask & (out["wetland_code"] != other)))
    del count, other
    system_of = np.array([0] + [SYSTEMS[a[0]] for a in codes[1:]], dtype="u1")
    out["wetland_system"] = system_of[out["wetland_code"]]
    polys["wetlands"] = W
    # ---- riparian ----
    R = collect(srcs, "riparian", F, ["ATTRIBUTE"], stats)
    out["riparian"] = burn(R["geometry_tile"], [1] * len(R), F, "u1", 0)
    polys["riparian"] = R
    # ---- mapping projects: where NWI has mapped, and the year of the imagery it was mapped from ----
    for role, flag in (("projects", "nwi_mapped"), ("riparian_projects", "riparian_mapped")):
        P = collect(srcs, role, F, PROJECT_FIELDS, stats)
        stats[f"{role}_status"] = {str(k): int(v) for k, v in P["STATUS"].astype(str).value_counts().items()}
        P = P[[is_digital(s) for s in P["STATUS"]]] if len(P) else P
        P = P.assign(year=[project_year(v) for v in P["IMAGE_YR"]])
        P = P.assign(order=[-1 if y is None else y for y in P["year"]])
        # the most recent imagery year is burnt last and wins where projects overlap; unknown years first
        P = P.sort_values(["order", "PROJECT_NAME"], kind="mergesort").reset_index(drop=True)
        out[flag] = burn(P["geometry_tile"], [1] * len(P), F, "u1", 0)
        polys[role] = P
        if role == "projects":
            vals = [FILL_I2 if y is None else y for y in P["year"]]
            out["image_year"] = burn(P["geometry_tile"], vals, F, "i2", FILL_I2)
            low = burn(P["geometry_tile"][::-1], vals[::-1], F, "i2", FILL_I2)
            stats["image_year_conflict_pixels"] = int(np.count_nonzero(mask & (out["image_year"] != low) & (low != FILL_I2)))
            have = out["image_year"][mask & (out["image_year"] != FILL_I2)]
            stats["image_year_range"] = [int(have.min()), int(have.max())] if have.size else []
            stats["n_projects"] = int(len(P))
    return out, dict(codes=codes, types=[types.get(a, "") for a in codes]), stats, polys


def spot_check(F, mask, out, polys, n, seed=0):
    """An independent check of the burn: for random aquifer pixels the centre is projected with GDAL
    (rasterio.warp.transform, not pyproj) into each source's CRS and tested with shapely's point-in-polygon
    against FWS's polygons as read (not densified, not reprojected); the rule for overlaps is applied to the
    hits. Centres within EDGE_TOL_M of a polygon edge are not judged (either answer is then right)."""
    rr, cc = np.nonzero(mask)
    if not rr.size:
        return 0
    pick = np.random.default_rng(seed).choice(rr.size, min(n, rr.size), replace=False)
    r, c = rr[pick], cc[pick]
    x, y = F["x0"] + (c + 0.5) * PIXEL, F["y0"] - (r + 0.5) * PIXEL
    judged = np.ones(r.size, bool)
    last = {}                                     # role -> index of the last-burnt polygon holding each centre
    for role, T in polys.items():
        top = np.full(r.size, -1, dtype=np.int64)
        for wkt, grp in T.groupby("src_crs"):
            sx, sy = rasterio.warp.transform(F["crs"], rasterio.crs.CRS.from_wkt(wkt), x.tolist(), y.tolist())
            pts = shapely.points(np.asarray(sx), np.asarray(sy))
            geoms = grp["geometry_src"].values.astype(object)
            p_i, g_i = shapely.STRtree(geoms).query(pts, predicate="within")
            np.maximum.at(top, p_i, grp.index.values[g_i])
            near, _ = shapely.STRtree(shapely.boundary(geoms)).query(pts, predicate="dwithin", distance=EDGE_TOL_M)
            judged[near] = False
        last[role] = top
    W, P = polys["wetlands"], polys["projects"]
    t = last["wetlands"]
    expect = dict(
        wetland_code=np.where(t >= 0, W["code"].values[np.maximum(t, 0)] if len(W) else 0, 0),
        riparian=(last["riparian"] >= 0).astype("u1"),
        nwi_mapped=(last["projects"] >= 0).astype("u1"),
        riparian_mapped=(last["riparian_projects"] >= 0).astype("u1"),
        image_year=np.where(last["projects"] >= 0,
                            np.array([FILL_I2 if v is None else v for v in P["year"]] or [FILL_I2])[np.maximum(last["projects"], 0)],
                            FILL_I2))
    bad = {v: int(np.count_nonzero(judged & (out[v][r, c] != e))) for v, e in expect.items()}
    if any(bad.values()):
        raise DataIntegrityError(f"spot check: of {int(judged.sum())} pixels these differ from point-in-polygon: {bad}")
    return int(judged.sum())


VAR_FILL = dict(wetland_system=FILL_U1, wetland_code=FILL_U2, riparian=FILL_U1, nwi_mapped=FILL_U1,
                riparian_mapped=FILL_U1, image_year=FILL_I2)
VAR_TYPE = dict(wetland_system="u1", wetland_code="u2", riparian="u1", nwi_mapped="u1", riparian_mapped="u1", image_year="i2")


def filled(out, mask):
    return {v: np.where(mask, out[v], np.array(VAR_FILL[v], dtype=VAR_TYPE[v])).astype(VAR_TYPE[v]) for v in VAR_FILL}


def write_tile(path, key, F, mask, arrays, tables, stats, srcs, checked):
    zkw = dict(zlib=COMPRESS_LEVEL > 0, complevel=max(COMPRESS_LEVEL, 1), shuffle=True)
    cy, cx = min(CHUNK_YX, F["h"]), min(CHUNK_YX, F["w"])
    crs = pyproj.CRS.from_user_input(F["crs"])
    tmp = path + ".new"
    with nc4.Dataset(tmp, "w", format="NETCDF4") as ds:
        ds.createDimension("y", F["h"])
        ds.createDimension("x", F["w"])
        ds.createDimension("code", len(tables["codes"]))
        yv = ds.createVariable("y", "f8", ("y",))
        yv.standard_name, yv.units, yv.axis, yv.long_name = "projection_y_coordinate", "m", "Y", "northing of pixel centre"
        yv[:] = F["y0"] - (np.arange(F["h"]) + 0.5) * PIXEL
        xv = ds.createVariable("x", "f8", ("x",))
        xv.standard_name, xv.units, xv.axis, xv.long_name = "projection_x_coordinate", "m", "X", "easting of pixel centre"
        xv[:] = F["x0"] + (np.arange(F["w"]) + 0.5) * PIXEL
        gm = ds.createVariable("crs", "i4")
        for k, v in crs.to_cf().items():
            gm.setncattr(k, v if isinstance(v, (str, int, float)) else np.asarray(v) if isinstance(v, (list, tuple)) else str(v))
        gm.spatial_ref = crs.to_wkt()
        gm.GeoTransform = f"{F['x0']} {PIXEL} 0.0 {F['y0']} 0.0 {-PIXEL}"
        gm.long_name = f"the 30 m grid of the Landsat, OpenET and CDL files, UTM zone {int(key[1:3])}, tile {key}"
        mv = ds.createVariable("roi_mask", "i1", ("y", "x"), chunksizes=(cy, cx), **zkw)
        mv.long_name = f"1 where the pixel centre lies inside the {ROI_LABEL} aquifer polygon"
        mv.flag_values, mv.flag_meanings, mv.grid_mapping = np.array([0, 1], dtype="i1"), "outside_roi inside_roi", "crs"
        mv[:] = mask.astype("i1")

        def var(name, long_name, **attrs):
            v = ds.createVariable(name, VAR_TYPE[name], ("y", "x"), fill_value=np.array(VAR_FILL[name], dtype=VAR_TYPE[name]),
                                  chunksizes=(cy, cx), **zkw)
            v.long_name, v.grid_mapping = long_name, "crs"
            for k, a in attrs.items():
                v.setncattr(k, a)
            v[:] = arrays[name]
            return v

        var("wetland_system", "Cowardin system of the NWI polygon holding the pixel centre",
            flag_values=np.arange(6, dtype="u1"), flag_meanings=" ".join(SYSTEM_NAMES[k] for k in range(6)),
            comment="0: no NWI wetland or deepwater polygon holds the pixel centre (see nwi_mapped for whether NWI "
                    "mapped the place at all). The system is the first letter of the polygon's ATTRIBUTE code. "
                    "_FillValue (255): outside the aquifer polygon.")
        var("wetland_code", "index into attribute_table: the NWI ATTRIBUTE code of the polygon holding the pixel centre",
            units="1", comment="0: no polygon. Where polygons overlap the smaller one is taken. _FillValue (65535): "
                               "outside the aquifer polygon. The table differs from tile to tile.")
        t = ds.createVariable("attribute_table", str, ("code",))
        t.long_name = "NWI ATTRIBUTE code (Cowardin system, subsystem, class, modifiers) of each wetland_code"
        t[:] = np.array(tables["codes"], dtype=object)
        t = ds.createVariable("wetland_type_table", str, ("code",))
        t.long_name = "FWS's WETLAND_TYPE label of each wetland_code"
        t[:] = np.array(tables["types"], dtype=object)
        var("riparian", "1 where an NWI riparian polygon holds the pixel centre", flag_values=np.array([0, 1], dtype="u1"),
            flag_meanings="not_riparian riparian",
            comment="FWS maps riparian areas only where riparian_mapped is 1; 0 elsewhere means not mapped, not "
                    "known to be absent. _FillValue (255): outside the aquifer polygon.")
        var("nwi_mapped", "1 where an NWI wetlands mapping project (status Digital) covers the pixel centre",
            flag_values=np.array([0, 1], dtype="u1"), flag_meanings="not_mapped mapped",
            comment="From the Wetlands_Project_Metadata layer. _FillValue (255): outside the aquifer polygon.")
        var("riparian_mapped", "1 where an NWI riparian mapping project (status Digital) covers the pixel centre",
            flag_values=np.array([0, 1], dtype="u1"), flag_meanings="not_mapped mapped",
            comment="From the Riparian_Project_Metadata layer. _FillValue (255): outside the aquifer polygon.")
        var("image_year", "year of the imagery the NWI mapping of the pixel was made from", units="1",
            comment="IMAGE_YR of the mapping project covering the pixel centre; where projects overlap, the most "
                    "recent. _FillValue (-32768): outside the aquifer polygon, not mapped, or year not stated.")
        ds.Conventions = "CF-1.9"
        ds.title = f"National Wetlands Inventory on the 30 m Landsat/OpenET/CDL grid, {ROI_LABEL} aquifer, tile {key}"
        ds.summary = ("FWS National Wetlands Inventory polygons (wetlands and deepwater habitats, riparian areas, and the "
                      "mapping projects with their imagery year) placed on the 30 m UTM tiles of this project's Landsat, "
                      "OpenET and CDL files: each pixel takes the polygon holding its centre. NWI is one snapshot "
                      "assembled from mapping projects of many years; image_year says which year each pixel shows.")
        ds.institution = "U.S. Fish and Wildlife Service (source data); packaged by the MSU groundwater project"
        ds.source = ("FWS National Wetlands Inventory, " + ("state geodatabases" if SOURCE_KIND == "state" else
                                                            "HU8 watershed geodatabases") + "; see the attribute sources")
        ds.references = ("Cowardin, L.M., Carter, V., Golet, F.C., LaRoe, E.T. (1979) Classification of wetlands and "
                         "deepwater habitats of the United States, FWS/OBS-79/31; FGDC Wetlands Classification Standard "
                         "FGDC-STD-004-2013; https://www.fws.gov/program/national-wetlands-inventory/data-download")
        ds.history = f"{iso(utcnow())} created by NWI_Wetlands_30m_Ogallala.ipynb"
        ds.tile, ds.roi = key, ROI_LABEL
        ds.frame = json.dumps(F)
        ds.sources = json.dumps([dict(file=s["name"], url=s["url"], size=s["size"], last_modified=s["last_modified"],
                                      md5=s["md5"], layers=s["layers"]) for s in srcs])
        ds.settings_signature = SETTINGS_SIGNATURE
        ds.processing_status = "complete"
        ds.burn_rule = ("pixel centre inside the polygon (GDAL rasterize, all_touched off) after the polygon's edges are cut "
                        f"into pieces of at most {DENSIFY_M:g} m in FWS's CRS and its corners projected exactly with pyproj")
        ds.overlap_rule = ("wetlands: the smaller polygon; mapping projects: the most recent imagery year; every layer of a "
                           "source is cut to that source's own outline (state or HU8 watershed) first")
        ds.spot_check = f"{checked} random aquifer pixels equal to a point-in-polygon test against FWS's polygons"
        ds.tile_facts = json.dumps(stats, default=str)
    os.replace(tmp, path)


def check_tile(path, mask, arrays, tables):
    """Read every variable back from the closed file and compare it with what was computed."""
    with nc4.Dataset(path) as ds:
        ds.set_auto_maskandscale(False)
        if not np.array_equal(np.asarray(ds["roi_mask"][:]).astype(bool), mask):
            raise DataIntegrityError(f"{os.path.basename(path)}: roi_mask differs from what was made")
        for v, a in arrays.items():
            if not np.array_equal(np.asarray(ds[v][:]), a):
                raise DataIntegrityError(f"{os.path.basename(path)}: {v} differs from what was computed")
        if list(ds["attribute_table"][:]) != tables["codes"]:
            raise DataIntegrityError(f"{os.path.basename(path)}: attribute_table differs from what was computed")

In [ ]:
# ═══════════════ CELL 8: tile by tile: check what is done, fetch FWS's files, make, check, Drive; run ═══════════════
# The unit of work is one tile: one file, built on scratch, checked, and copied to Drive under its final name.
# FWS's files are fetched once per run and serve every tile. A runtime that dies loses the tile it was working
# on; the next run skips every tile that is on Drive, complete, and made from the same files and settings.
SETTINGS_SIGNATURE = json.dumps(dict(
    product=PRODUCT_TAG, version=1, source=SOURCE_KIND, states=STATES if SOURCE_KIND == "state" else None,
    pixel=PIXEL, tile_px=TILE_PX, test_frame_px=TEST_FRAME_PX if TEST_MODE else None, densify_m=DENSIFY_M,
    systems=SYSTEMS, burn="pixel centre", wetland_overlap="smaller polygon", project_overlap="latest image year",
    project_status="Digital", cut_to_source_outline=True), sort_keys=True)


def unit_path(key):
    return os.path.join(DRIVE_OUT, key, f"{PRODUCT_TAG}_{key}_{ROI_LABEL}.nc")


def file_state(path):
    """(status, signature, sources) of a tile file on Drive; None when there is none. A file Drive cannot
    open is tried again and then stops the run: it is never replaced because Drive failed to read it."""
    if not os.path.exists(path):
        return None
    why = None
    for attempt in range(3):
        try:
            with nc4.Dataset(path) as ds:
                return (str(getattr(ds, "processing_status", "")), str(getattr(ds, "settings_signature", "")),
                        json.loads(str(getattr(ds, "sources", "[]"))))
        except Exception as e:
            why = e
            time.sleep(20 * (attempt + 1))
    raise PipelineStop(f"{os.path.basename(path)} is on Drive but cannot be read ({why}). Nothing was changed. If it "
                       "really is damaged, move it out of the folder yourself and run again.")


def done_on_drive(path, man):
    """True when the tile's file is on Drive, complete, made with these settings, and every FWS file it was
    made from is the one in sources.json (same size, date and MD5)."""
    st = file_state(path)
    if st is None:
        return False
    status, signature, sources = st
    if status != "complete" or signature != SETTINGS_SIGNATURE or not sources:
        return False
    for s in sources:
        m = man.get(s["file"])
        if not m or (m["size"], m["last_modified"], m.get("md5")) != (s["size"], s["last_modified"], s["md5"]):
            return False
    return True


def process_tile(key, F, names, srcs, prog):
    t0 = time.time()
    mask = frame_mask(key, F)
    use = sources_of_tile(F, names, srcs)
    if not use:
        raise DataIntegrityError(f"{key}: none of the FWS files read ({', '.join(names)}) reaches this tile")
    out, tables, stats, polys = make_tile(key, F, use, mask)
    with timed("spot check"):
        checked = spot_check(F, mask, out, polys, SPOT_CHECK_PX, seed=int(hashlib.md5(key.encode()).hexdigest()[:8], 16))
    del polys
    arrays = filled(out, mask)
    work = os.path.join(NC_DIR, f"{key}.nc")
    with timed("write"):
        write_tile(work, key, F, mask, arrays, tables, stats, use, checked)
        check_tile(work, mask, arrays, tables)
    dst = unit_path(key)
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    robust_drive_copy(work, dst)
    size = os.path.getsize(work)
    md5 = md5sum(work)
    os.remove(work)
    n = max(1, int(mask.sum()))
    share = {SYSTEM_NAMES[k]: round(100 * np.count_nonzero(mask & (out["wetland_system"] == k)) / n, 3) for k in range(1, 6)}
    log_events(LEDGER_NAME, [dict(file=os.path.basename(dst), tile=key, size=size, md5=md5, n_roi=int(mask.sum()),
                                  sources=[s["name"] for s in use], wetland_percent=share)])
    prog[0] += 1
    log.info("[%s] DONE: %s on Drive | %d sources, %d wetland polygons | wetland %% of aquifer pixels %s | "
             "riparian %.2f %% | mapped %.2f %% | imagery %s | %d pixels checked | %.0f s",
             key, human(size), len(use), sum(v.get("wetlands", 0) for v in stats["features_read"].values()),
             {k: v for k, v in share.items() if v}, 100 * np.count_nonzero(mask & (out["riparian"] == 1)) / n,
             100 * np.count_nonzero(mask & (out["nwi_mapped"] == 1)) / n, stats.get("image_year_range"),
             checked, time.time() - t0)
    gc.collect()


def run():
    t0 = time.time()
    TOTALS.clear()
    TIMES.clear()
    stopped = None
    try:
        ensure_disk_free("start")
        frames = get_frames()
        if ONLY_TILES:
            frames = {k: v for k, v in frames.items() if k in set(ONLY_TILES)}
        if TEST_MODE:
            missing = sorted(set(TEST_TILES) - set(frames))
            if missing:
                raise PipelineStop(f"TEST_TILES {missing} have no frame (no aquifer pixel, or not in frames.json).")
        files, per_tile = wanted_files(frames)
        man = check_release(files)
        todo = {k: F for k, F in sorted(frames.items()) if not done_on_drive(unit_path(k), man)}
        log.info("NWI on %d tiles of the 30 m grid, from %d FWS %s files -> %s | %d tiles to make",
                 len(frames), len(files), SOURCE_KIND, DRIVE_OUT, len(todo))
        if todo:
            names = sorted({n for k in todo for n in per_tile[k]})
            srcs = prepare_sources(names, man)
            prog = [0]
            for key, F in todo.items():
                process_tile(key, F, per_tile[key], srcs, prog)
            if not KEEP_DOWNLOADS:
                for n in names:
                    shutil.rmtree(os.path.join(RAW_DIR, n[:-4]), ignore_errors=True)
    except PipelineStop as e:
        stopped = str(e)
    except KeyboardInterrupt:
        stopped = "Stopped with the stop button."
    log.info("=" * 78)
    if stopped:
        print("\n" + "=" * 78 + f"\n  STOPPED (nothing on Drive is lost, re-run to resume)\n  {stopped}\n" + "=" * 78 + "\n")
    log.info("  this run: %s downloaded from FWS, %.1f min | %s", human(TOTALS.get("bytes", 0)), (time.time() - t0) / 60,
             ", ".join(f"{k} {v:.0f} s" for k, v in sorted(TIMES.items(), key=lambda kv: -kv[1])))


run()

In [ ]:
# ═══════════════ CELL 9: what is on Drive ═══════════════
def audit():
    man = load_manifest()
    path = os.path.join(DRIVE_OUT, FRAMES_NAME)
    frames = {}
    if os.path.exists(path):
        with open(path) as f:
            frames = {k: v for k, v in json.load(f)["frames"].items() if v}
    if ONLY_TILES:
        frames = {k: v for k, v in frames.items() if k in set(ONLY_TILES)}
    print(f"{DRIVE_OUT}\n\n{'tile':10s} {'size':>9s} {'aquifer px':>11s} {'wetland %':>9s} {'riparian %':>10s} "
          f"{'mapped %':>8s} {'imagery':>11s}  state")
    total, missing = 0, []
    for key in sorted(frames):
        p = unit_path(key)
        if not os.path.exists(p):
            missing.append(key)
            continue
        size = os.path.getsize(p)
        total += size
        try:
            ok = done_on_drive(p, man)
            with nc4.Dataset(p) as ds:
                ds.set_auto_maskandscale(False)
                roi = np.asarray(ds["roi_mask"][:]) == 1
                n = max(1, int(roi.sum()))
                wet = 100 * np.count_nonzero(roi & (np.asarray(ds["wetland_system"][:]) > 0)) / n
                rip = 100 * np.count_nonzero(roi & (np.asarray(ds["riparian"][:]) == 1)) / n
                mapped = 100 * np.count_nonzero(roi & (np.asarray(ds["nwi_mapped"][:]) == 1)) / n
                yr = json.loads(ds.tile_facts).get("image_year_range") or ["-", "-"]
            state = "complete" if ok else "STALE (other settings or FWS files: remade by the next run)"
            print(f"{key:10s} {human(size):>9s} {int(roi.sum()):11d} {wet:9.2f} {rip:10.2f} {mapped:8.2f} "
                  f"{str(yr[0]) + '-' + str(yr[1]):>11s}  {state}")
        except Exception as e:
            print(f"{key:10s} {human(size):>9s}  UNREADABLE ({str(e)[:80]})")
    print(f"\n{len(frames) - len(missing)} of {len(frames)} tiles on Drive, {human(total)}"
          + (f"; missing: {', '.join(missing)}" if missing else ""))
    if man:
        print("FWS files of this folder:", ", ".join(f"{n} ({v['last_modified']})" for n, v in sorted(man.items())))


audit()

# Reading a tile:
#   import xarray as xr, json
#   root = "/content/drive/MyDrive/MSUGWB/NWI_wetlands_30m_tiles"
#   ds = xr.open_dataset(f"{root}/z14e02n29/NWI_wetlands_30m_tiles_z14e02n29_Ogallala.nc")
#   wet = ds.wetland_system.where(ds.roi_mask == 1) > 0        # any NWI wetland or deepwater polygon
#   codes = ds.attribute_table.values[ds.wetland_code.fillna(0).astype(int)]   # e.g. "PEM1C"
#   unmapped = ds.nwi_mapped == 0                               # inside the aquifer but never mapped by NWI
# The tiles have the names, x and y of the Landsat, OpenET and CDL files, so all of them line up pixel for pixel.